# 07_high_target_variance

Generated from `alteryx-workflows\07_high_target_variance.yxmd` by compiler 2.2.0.

## How to use this notebook

1. Review and complete the source configuration.
2. Run cells from top to bottom.
3. Inspect the output previews and validate semantic parity.

> **Important:** Generated transformations are unvalidated. Confirm schemas, row counts, business rules, and destinations before promotion.


## Runtime imports

Imports and runtime helpers required by the generated PySpark transformations.


In [ ]:
from pyspark.sql import Window, functions as F


## Source ingestion

Loads and validates source datasets before transformation.


### Source ingestion 1 of 1

Produces `orders`, `targets`.


In [ ]:
orders = spark.createDataFrame([["C001", "West", "Online", "2026-01-12", "SKU-A-100", "1200", "4"], ["C002", "East", "Store", "2026-01-18", "SKU-B-220", "840", "2"], ["C001", "West", "Online", "2026-02-03", "SKU-A-140", "640", "1"], ["C003", "North", "Partner", "2026-02-11", "SKU-C-010", "2100", "6"], ["C004", "South", "Store", "2026-03-01", "SKU-B-220", "430", "3"], ["C002", "East", "Online", "2026-03-09", "SKU-A-100", "990", "2"], ["C005", "West", "Partner", "2026-03-21", "SKU-C-010", "150", "1"], ["C003", "North", "Store", "2026-04-02", "SKU-A-140", "1760", "5"]], ["CustomerID", "Region", "Channel", "OrderDate", "SKU", "Amount", "Units"])
targets = spark.createDataFrame([["West", "5000"], ["East", "3000"], ["North", "4000"], ["South", "2000"]], ["Region", "TargetAmount"])


## Transformations

Applies the translated Alteryx operations in dependency order.


### Transformations 1 of 1

Produces `numeric_amount`, `chronological`, `running_total_by_region`, `region_totals`, and 4 more.


In [ ]:
numeric_amount = orders.withColumn("Amount", F.col("Amount").cast('double'))
chronological = numeric_amount.orderBy(F.col("OrderDate").asc())
running_total_by_region = chronological.withColumn("RunningTotal", F.sum(F.col("Amount").cast('double')).over(Window.partitionBy(*["Region"]).orderBy(F.col("OrderDate").asc()).rowsBetween(Window.unboundedPreceding, Window.currentRow)))
region_totals = running_total_by_region.groupBy(*["Region"]).agg(F.sum(F.col("Amount").cast('double')).alias("Sum_Amount"), F.count(F.col("Amount")).alias("RowCount"))
numeric_target = targets.withColumn("TargetAmount", F.col("TargetAmount").cast('double'))
actual_vs_target = region_totals.join(numeric_target, (region_totals["Region"] == numeric_target["Region"]), 'inner').drop(numeric_target["Region"])
variance = actual_vs_target.withColumn("Variance", F.col("Sum_Amount") - F.col("TargetAmount"))
status = variance.withColumn("Status", F.when(F.col("Variance") >= 0, F.lit("Above")).otherwise(F.lit("Below")))


## Output preview

Displays terminal DataFrames for review. Replace preview-only output with approved destination writes after validation.


### Output preview 1 of 1


In [ ]:
display(status)
display(running_total_by_region)
